In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
Dir_dataset = '/work/desai-lab/ABC_dataset'
Dir_github = '/work/xy6/Project/ABC/github'

Dir_fmriprep = os.path.join(Dir_dataset,'derivatives','fmriprep')
repo = 'ABC_valence'
pipeline = 'afni_smooth-bold'
Dir_deriv = os.path.join(Dir_github,'derivatives')
Dir_working = os.path.join(Dir_github,'derivatives','fmri',pipeline)
df_master = pd.read_csv(os.path.join(Dir_dataset,'participants_master.csv'))
df_master

,subid,visit,Passage_eprime,story,Fam_eprime,T1,Passage_fmri,Fam_fmri,rest_fmri,subject_id,confirm_abcid,sex,age_dayofday1testing,handedness,first_language,other_language
0,1001,1.0,NaN,NaN,NaN,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,ABC1001,ABC1001,2.0,65.0,2.0,1.0,NaN
1,1002,1.0,NaN,NaN,NaN,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,ABC1002,ABC1002,1.0,61.0,2.0,1.0,NaN
2,1002,2.0,1.0,Jacob_HappyDay.wav,NaN,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,ABC1002,ABC1002,1.0,61.0,2.0,1.0,NaN
3,1002,5.0,1.0,Brandi_HappyDay.wav,NaN,NaN,NaN,NaN,NaN,ABC1002,ABC1002,1.0,61.0,2.0,1.0,NaN
4,1003,1.0,NaN,NaN,NaN,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,ABC1003,ABC1003,2.0,75.0,2.0,1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
327,10393,2.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
328,10425,5.0,1.0,Allison_MetaphorsPronouns.wav,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
329,10445,1.0,1.0,Jack_Roswell.wav,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
330,10535,1.0,1.0,Jack_MetaphorsPronouns.wav,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df_subj = df_master.copy()
df_subj = df_subj.loc[~df_subj['Passage_eprime'].isna()].loc[~df_subj['Passage_fmri'].isna()]
df_subj = df_subj.loc[df_subj['visit']==1]
# df_subj = df_subj[['subid','visit','Passage_fmri']]
df_subj[['subid','visit','Passage_fmri']]

,subid,visit,Passage_fmri
11,1006,1.0,/work/desai-lab/ABC_dataset/derivatives/fmripr...
12,1007,1.0,/work/desai-lab/ABC_dataset/derivatives/fmripr...
13,1008,1.0,/work/desai-lab/ABC_dataset/derivatives/fmripr...
14,1009,1.0,/work/desai-lab/ABC_dataset/derivatives/fmripr...
18,1012,1.0,/work/desai-lab/ABC_dataset/derivatives/fmripr...
...,...,...,...
282,1318,1.0,/work/desai-lab/ABC_dataset/derivatives/fmripr...
283,1320,1.0,/work/desai-lab/ABC_dataset/derivatives/fmripr...
284,1321,1.0,/work/desai-lab/ABC_dataset/derivatives/fmripr...
285,1322,1.0,/work/desai-lab/ABC_dataset/derivatives/fmripr...


In [6]:
FWHM = 6
List_tasks = ['Passage']
Dir_script = os.path.join(Dir_working,'scripts')
if not os.path.exists(Dir_script):
    os.makedirs(Dir_script)
    
for i,row in df_subj.iterrows():
    subid = int(row['subid'])
    visit = int(row['visit'])
    scriptFile = os.path.join(Dir_script,'batch_smooth_{}.sh'.format(subid))
    with open(scriptFile, "w") as f:
        f.write("#!/bin/tcsh\n")

    Dir_fmriprep_subj = os.path.join(Dir_fmriprep,f'sub-{subid}',f'ses-visit{visit:02}')
    if not os.path.exists(Dir_fmriprep_subj):
        print(f'ses-visit{visit:02}: no fmri data.')
        continue

    Dir_smooth_subj = os.path.join(Dir_working,f'sub-{subid}')
    if not os.path.exists(Dir_smooth_subj):
        os.makedirs(Dir_smooth_subj,exist_ok=True)

    for task in List_tasks:
        img_fmri = os.path.join(Dir_fmriprep_subj,'func',
                              f"sub-{subid}_ses-visit{visit:02}_task-{task}_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz")
        img_mask = os.path.join(Dir_fmriprep_subj,'func',
                              f'sub-{subid}_ses-visit{visit:02}_task-{task}_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz')
        if not os.path.exists(img_fmri):
            print(f'ses-visit{visit:02}: no fmri {task} data.')
            continue
        if not os.path.exists(img_mask):
            print(f'ses-visit{visit:02}: no fmri {task} mask.')
            continue
        
        smoothFile = os.path.join(Dir_smooth_subj,
                          f'sub-{subid}_ses-visit{visit:02}_task-{task}_space-MNI152NLin2009cAsym_res-2_desc-sm{FWHM}_bold.nii.gz')
        with open(scriptFile, "a") as f:
            f.write(f"echo start smoothing sub-{subid} ses-visit{visit:02} task-{task}\n")
            f.write(f'/work/apps/AFNI/linux_openmp_64/3dBlurToFWHM -mask {img_mask} -FWHM {FWHM} \\\n')
            f.write(f'-input {img_fmri} -prefix {smoothFile} -blurmaster {img_fmri} -detrend -bmall \n')
        

In [7]:
df_ana = df_subj.loc[df_subj['subid']>1197]
text_subj = ','.join(df_ana['subid'].astype(str).values)
print(f"N={df_ana.shape[0]}")
text_subj

N=60


'1199,1200,1204,1206,1208,1209,1210,1212,1221,1223,1226,1227,1229,1233,1234,1238,1239,1243,1245,1246,1247,1250,1251,1252,1255,1257,1259,1264,1267,1268,1270,1271,1272,1274,1276,1277,1279,1281,1283,1284,1285,1286,1287,1293,1294,1295,1297,1299,1300,1301,1302,1305,1306,1309,1311,1318,1320,1321,1322,1323'

In [2]:
a = [1199,1200,1204,1206,1208,1209,1210,1212,1221,1223,1226,1227,1229,1233,1234,1238,1239,1243,1245,1246,1247,1250,1251,1252,1255,1257,1259,1264,1267,1268,1270,1271,1272,]
len(a)

33

In [9]:
df_ana = df_subj.loc[df_subj['subid']<=1197]
text_subj = ','.join(df_ana['subid'].astype(str).values)
print(f"N={df_ana.shape[0]}")
text_subj

N=112


'1006,1007,1008,1009,1012,1014,1016,1018,1019,1020,1021,1022,1023,1024,1025,1026,1028,1029,1030,1032,1036,1037,1039,1040,1041,1042,1044,1045,1047,1048,1049,1052,1053,1054,1055,1056,1057,1058,1059,1060,1061,1065,1066,1068,1069,1070,1072,1073,1074,1076,1079,1083,1085,1086,1087,1088,1089,1090,1092,1095,1097,1098,1099,1100,1102,1104,1105,1106,1107,1108,1109,1112,1118,1123,1124,1125,1126,1130,1134,1136,1137,1138,1140,1142,1145,1146,1147,1148,1149,1150,1151,1153,1154,1155,1156,1159,1160,1161,1165,1166,1171,1173,1174,1175,1177,1180,1182,1183,1185,1187,1194,1197'

In [12]:
Dir_logs = os.path.join(Dir_working,'logs')
if not os.path.exists(Dir_logs):
    os.makedirs(Dir_logs)

jobname = pipeline
ncores = 6
# partition = 'desailab-48core'
partition = 'defq-64core'
script_slurm = os.path.join(Dir_script,f'Batch2_slurm_{pipeline}.sh')
# array = '001-013'
array = text_subj 
#array = '001-013'
with open(script_slurm, "w") as f:
    f.write("#!/bin/sh\n")
    f.write("### name your job \n")
    f.write(f"#SBATCH --job-name={jobname} \n")
    f.write("### specify number of cores to use \n")
    f.write(f"#SBATCH -n {ncores} \n")
    f.write(f"#SBATCH -N {1} \n")
    f.write("### which partition do you want to use? \n")
    f.write(f"#SBATCH -p {partition} \n")
    f.write("### %a specifies that the output file for each subject will be named according to their array ID \n")
    f.write("# Where to output log files? \n")
    f.write(f"#SBATCH --output='{Dir_logs}/{pipeline}-%A_%a.log' \n")
    f.write(f"#SBATCH --error='{Dir_logs}/{pipeline}-%A_%a.err' \n")
    f.write("### list the subjects that you want to submit in this array \n")
    f.write(f"#SBATCH --array={array} \n")
    f.write("module load afni \n")
    f.write("module load python3/anaconda/2021.11 \n")
    f.write('echo "Slurm job ID: " $SLURM_JOB_ID \n')
    f.write('echo "Slurm array task ID: " $SLURM_ARRAY_TASK_ID \n')

    f.write('printf -v subj "%03d" $SLURM_ARRAY_TASK_ID \n')
    f.write('echo "Running 3dTproject on sub-$subj" \n')
    f.write(f'cd {Dir_script} \n')
    f.write('tcsh batch_smooth_$subj.sh \n')
    f.write(f'echo "Finished {pipeline} sub-$subj" \n')
    f.write('echo date \n')
    